In [5]:
%load_ext autoreload
%autoreload 2
%reset -f

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Imports

In [6]:
from locallib.picarrodb import *
from locallib.query import *
from locallib.box import *
from locallib.query import *

from datetime import datetime
import geopandas as gpd
from shapely import wkt
import matplotlib.pyplot as plt
import contextily as ctx
import pandas as pd
import h3
from shapely import wkt
import matplotlib.pyplot as plt
import geopandas as gpd
import contextily as ctx
from shapely.geometry import Polygon
import pandas as pd

from shapely import wkt
from shapely.geometry import LineString, MultiLineString
import sqlite3

import logging

In [7]:

# Set up logging with file and stream handlers
log_filename = 'DA-3507.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s:%(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(log_filename, mode='a')
    ]
)
logger = logging.getLogger(__name__)

In [8]:
customer_name = "Cadent"
start_date = "2025-01-01"
end_date = "2026-04-24"

In [9]:

# Create a SQLite database called picarro_KPI.db in the database directory, with the requested table and fields
import sqlite3
import os

db_path = os.path.join('database', 'picarro_kpi.db')
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Create SurveyH3 table if it does not exist
cursor.execute("""
CREATE TABLE IF NOT EXISTS SurveyH3Boundary (
    ID INTEGER PRIMARY KEY AUTOINCREMENT, -- Unique identifier
    SurveyId VARCHAR(255), -- UUID key
    Resolution INTEGER,
    H3_cell BIGINT,
    Counts INTEGER,
    Passes INTEGER
)
""")

conn.commit()
conn.close()


## Get the surveys

In [10]:
output_log = "="*20+'\n'
output_log = f'Starting the process for {customer_name}  \n '
output_log += f'Process started at: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n'
#Add current date to the output log as a header
output_log += "="*20+'\n'
#Query the data from the EU2 Database
a = get_users(customer_name, '#UserList')
a.set_child(get_surveys('#UserList',start_date = start_date,end_date = end_date))
current_data = a.execute(EU2_Conn)
print("Number of surveys: ", len(current_data))

Number of surveys:  6863


## Get the breadcrumb trajectory 


In [11]:
# Example SQL query that returns a STUnion geometry as WKT text for segments whose SurveyId matches those in #TempSurvey, using MSSQL format:
segment_union_query = """
SELECT 
    SurveyId,
    geometry::UnionAggregate(Shape).STAsText() AS AggBreadcrumbs
FROM Segment
WHERE SurveyId IN (SELECT SurveyId FROM #TempSurvey)
GROUP BY SurveyId
"""

current_data.db.set_query(segment_union_query)
segment_union = current_data.db.execute(EU2_Conn, temp_table_name = '#TempSurvey', source_col = 'SurveyId')
survey = pd.merge(current_data, segment_union, on = 'SurveyId', how = 'left')

# Rasterize the boundary

In [12]:
# Choose an H3 resolution (for example, 8 for neighborhood scale)
h3_resolution = 11
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

for b in range(len(survey)):
    try:
        logging.info(f"Processing survey {b} of {len(survey)}, Survey id: {survey.iloc[b]['SurveyId']}")
        # Convert the WKT geometry in 'geom' column of the first row to a Shapely object
        boundary_geom = wkt.loads(survey.iloc[b]['SurveyArea'])

        # H3 v4+: polygon_to_cells expects LatLngPoly/LatLngMultiPoly, not a GeoJSON dict.
        # geo_to_cells accepts Shapely (via __geo_interface__) or a GeoJSON dict.
        h3_cells = h3.geo_to_cells(boundary_geom, h3_resolution)

        # List of cell index strings (order not guaranteed by h3)
        h3_cells_list = list(h3_cells)

        def h3_cell_polygon(cell):
            # Returns a shapely Polygon for an h3 cell index (in degrees)
            boundary = h3.cell_to_boundary(cell)
            # Invert lat and long (swap their positions in each tuple)
            boundary = [(lat, lon) for lon, lat in boundary]
            return Polygon(boundary)

        # Create polygons for each H3 cell
        h3_polys = [h3_cell_polygon(cell) for cell in h3_cells_list]
        h3_gdf = gpd.GeoDataFrame({'h3_cell': h3_cells_list, 'geometry': h3_polys}, crs='EPSG:4326')
        # Project h3_gdf to a projected CRS (e.g., Web Mercator) before buffering
        h3_gdf_proj = h3_gdf.to_crs(epsg=3857)
        h3_gdf_proj['offset'] = h3_gdf_proj['geometry'].buffer(-2)  # 2 meters in projected CRS (meters)

        # Get the aggregated breadcrumbs geometry (WKT or list of WKT)
        agg_breadcrumbs = survey.iloc[b]["AggBreadcrumbs"]

        geom = wkt.loads(agg_breadcrumbs)

        # Make GeoDataFrame for aggregated breadcrumbs (assume they're still in EPSG:4326 to match original data)
        agg_breadcrumbs_gdf = gpd.GeoDataFrame({"geometry": geom}, crs='EPSG:4326')
        # Project them to match the projected CRS used for polygons/h3_gdf_proj
        agg_breadcrumbs_gdf = agg_breadcrumbs_gdf.to_crs(epsg=3857)

        # Create GeoDataFrame for the lines of the polygons (the boundaries of H3 cells)
        # (Already in EPSG:3857)
        polygon_lines = h3_gdf_proj.offset.boundary
        polygon_lines_gdf = gpd.GeoDataFrame({"geometry": polygon_lines}, crs='EPSG:3857')

        # Intersect the lines of the polygons with the breadcrumbs
        intersection_gdf = gpd.overlay(polygon_lines_gdf, agg_breadcrumbs_gdf, how='intersection', keep_geom_type=False)

        # For each MultiPoint geometry in intersection_gdf, convert points to lat/lon and then to h3 cells
        multipoint_to_h3_cells = []  # List to hold (orig_index, list_of_h3_cells) for each MultiPoint
        h3_cells = []
        intersection_gdf.to_crs(epsg=3857)
        # Ensure intersection_gdf is in EPSG:4326 (lat/lon) before accessing coordinates to pass to h3
        intersection_latlon = intersection_gdf.to_crs(epsg=4326)

        for idx, geom in enumerate(intersection_latlon['geometry']):
            if geom.geom_type == "MultiPoint":
                for pt in geom.geoms:
                    # pt.x is longitude, pt.y is latitude in EPSG:4326
                    h3_cell = h3.latlng_to_cell(pt.y, pt.x, h3_resolution)
                    h3_cells.append(h3_cell)

        h3_cells_df = pd.DataFrame({'h3_cell': h3_cells})
        h3_cells_counts_df = h3_cells_df.value_counts().reset_index(name='Counts')
        h3_cells_counts_df['Passes'] = h3_cells_counts_df['Counts']/2

        h3_cells_counts_df['SurveyId'] = survey.iloc[b]['SurveyId']
        h3_cells_counts_df['Resolution'] = h3_resolution
        # Convert h3_cell from hexadecimal string to its integer value
        h3_cells_counts_df['h3_cell'] = h3_cells_counts_df['h3_cell'].apply(lambda x: int(str(x), 16))
        h3_cells_counts_df
 
 

    except Exception as e:
        logging.error(f"Error processing survey {b} of {len(survey)}, Survey id: {survey.iloc[b]['SurveyId']}: {e}")
        continue

    db_path = 'database/picarro_kpi.db'   # Adjusted the path as instructed

    # Try update first, if no rows affected insert instead (manual UPSERT)
    for _, row in h3_cells_counts_df.iterrows():
        cursor.execute("""
            UPDATE SurveyH3Boundary
            SET Counts=?, Passes=?
            WHERE h3_cell=? AND SurveyId=? AND Resolution=?
            """,
            (
                int(row['Counts']),
                float(row['Passes']),
                row['h3_cell'],
                row['SurveyId'],
                int(row['Resolution'])
            )
        )
        if cursor.rowcount == 0:
            cursor.execute("""
                INSERT INTO SurveyH3Boundary (h3_cell, Counts, Passes, SurveyId, Resolution)
                VALUES (?, ?, ?, ?, ?)
                """,
                (
                    row['h3_cell'],
                    int(row['Counts']),
                    float(row['Passes']),
                    row['SurveyId'],
                    int(row['Resolution'])
                )
            )

        conn.commit()
conn.close()

    

2026-05-01 18:49:06,581 INFO:Processing survey 0 of 6863, Survey id: 6CA440A4-F7FA-E1E2-EB4B-3A173AEF17FB
2026-05-01 18:49:10,869 INFO:Processing survey 1 of 6863, Survey id: 02737CA3-C731-4FC7-1FD0-3A173B2CD33E
2026-05-01 18:49:14,872 INFO:Processing survey 2 of 6863, Survey id: 56D91ADD-727E-D657-AA74-3A173BB0C4E0
2026-05-01 18:49:18,698 INFO:Processing survey 3 of 6863, Survey id: BD086E13-68CB-DF79-131D-3A173BB19974
2026-05-01 18:49:22,713 INFO:Processing survey 4 of 6863, Survey id: 4FB62A6B-0EC3-4F4D-1F45-3A1740075792


KeyboardInterrupt: 